### Prepare dialogue 

In [1]:
import pandas as pd

with open("../own_script/dialogue_1/dialogue_1.txt", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]

df = pd.DataFrame({
    "utterance_id": range(1, len(lines)+1),
    "text": lines,
})
df.head()

,utterance_id,text
0,1,I’m so tired! This player just keeps running a...
1,2,Honestly? A solid 9. It’s boring and feels lik...
2,3,"Both! A good player should stand their ground,..."
3,4,You mean you're saying they're running because...
4,5,"I see. When you put it like that, chasing them..."


### Load text/VAD model

In [2]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


torch: 2.10.0+cpu
transformers: 4.46.3


c:\Users\Legion 5 Pro\OneDrive\Documents\Graduate research\test\park\park_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "RobroKools/vad-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

### Predict function

In [4]:
import numpy as np

def predict_vad(texts):
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
    # logits shape: [batch, 3] = [V, A, D]
    vad = out.logits.cpu().numpy()
    return vad  # np.array [batch,3]


### Check value range of vac-bert

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tok = AutoTokenizer.from_pretrained("RobroKools/vad-bert")
model = AutoModelForSequenceClassification.from_pretrained("RobroKools/vad-bert")

samples = [
    "I feel terrible and hopeless.",
    "I feel completely neutral.",
    "I feel amazing and so happy!",
]

for s in samples:
    inputs = tok(s, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().tolist()
    print(s, "-> VAD:", out)


I feel terrible and hopeless. -> VAD: [1.7622524499893188, 3.2708568572998047, 2.4671969413757324]
I feel completely neutral. -> VAD: [2.781001567840576, 2.894531488418579, 3.203058958053589]
I feel amazing and so happy! -> VAD: [4.503335952758789, 4.0434064865112305, 3.532799482345581]


In [6]:
samples = [
    "I want to die. I hate everything.",
    "I feel completely empty and numb.",
    "This is fine.",
    "I'm a little annoyed.",
    "I'm so excited I can't stop screaming!",
    "I feel calm, peaceful, and relaxed.",
]

vals = []
for s in samples:
    inputs = tok(s, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().tolist()  # [V,A,D]
    print(s, "->", out)
    vals.append(out)

import numpy as np
vals = np.array(vals)
print("Valence range:", vals[:,0].min(), vals[:,0].max())
print("Arousal range:", vals[:,1].min(), vals[:,1].max())
print("Dominance range:", vals[:,2].min(), vals[:,2].max())


I want to die. I hate everything. -> [1.4248108863830566, 3.895531415939331, 2.7246110439300537]
I feel completely empty and numb. -> [1.9501692056655884, 3.131157159805298, 2.477699041366577]
This is fine. -> [3.1369645595550537, 3.0026485919952393, 3.1079323291778564]
I'm a little annoyed. -> [1.9209797382354736, 3.553025960922241, 2.9750216007232666]
I'm so excited I can't stop screaming! -> [3.0788471698760986, 4.616365909576416, 3.017502546310425]
I feel calm, peaceful, and relaxed. -> [3.758591413497925, 2.6541788578033447, 3.186662435531616]
Valence range: 1.4248108863830566 3.758591413497925
Arousal range: 2.6541788578033447 4.616365909576416
Dominance range: 2.477699041366577 3.186662435531616


### Test on dialogue 10 and save into dataframe

In [ ]:
vad = predict_vad(df["text"].tolist())
df["valence_text"] = vad[:, 0]
df["arousal_text"] = vad[:, 1]
df["dominance_text"] = vad[:, 2]

df.head()

,utterance_id,text,valence_text,arousal_text,dominance_text
0,1,I’m so tired! This player just keeps running a...,2.423500,3.971787,2.804366
1,2,Honestly? A solid 9. It’s boring and feels lik...,2.541238,3.633606,3.026369
2,3,"Both! A good player should stand their ground,...",2.839795,3.533370,3.609603
3,4,You mean you're saying they're running because...,2.743749,3.480893,3.065911
4,5,"I see. When you put it like that, chasing them...",2.702262,3.347695,3.295031


### Check with own dataset

In [8]:
import numpy as np

v = df["valence_text"].to_numpy()
a = df["arousal_text"].to_numpy()

print("Valence min/max:", v.min(), v.max())
print("Valence q1/median/q3:", np.quantile(v, [0.25, 0.5, 0.75]))

print("Arousal min/max:", a.min(), a.max())
print("Arousal q1/median/q3:", np.quantile(a, [0.25, 0.5, 0.75]))


Valence min/max: 2.4234996 3.2645586
Valence q1/median/q3: [2.58149374 2.72300577 2.8157835 ]
Arousal min/max: 3.2820504 3.971787
Arousal q1/median/q3: [3.3809945  3.50713158 3.60854709]


### Scale mismatch issue --> Normalization

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To solve this, I implement the normalization method to vac-bert scale to be compatible with wagner and the paper

In [9]:
import numpy as np

# กำหนดช่วง text สมมติเป็น 1..5
V_MIN, V_MAX = 1.0, 5.0

def to_minus1_1(x, xmin=V_MIN, xmax=V_MAX):
    return 2 * (x - xmin) / (xmax - xmin) - 1  # map [xmin,xmax] -> [-1,1]

df["valence_text_n"]  = to_minus1_1(df["valence_text"])
df["arousal_text_n"]  = to_minus1_1(df["arousal_text"])
df["dominance_text_n"] = to_minus1_1(df["dominance_text"])


In [10]:
df

,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n
0,1,I’m so tired! This player just keeps running a...,2.423500,3.971787,2.804366,-0.288250,0.485893,-0.097817
1,2,Honestly? A solid 9. It’s boring and feels lik...,2.541238,3.633606,3.026369,-0.229381,0.316803,0.013185
2,3,"Both! A good player should stand their ground,...",2.839795,3.533370,3.609603,-0.080103,0.266685,0.304801
3,4,You mean you're saying they're running because...,2.743749,3.480893,3.065911,-0.128125,0.240447,0.032956
4,5,"I see. When you put it like that, chasing them...",2.702262,3.347695,3.295031,-0.148869,0.173847,0.147515
5,6,"Yeah thanks, I will try this new tactic, maybe...",3.264559,3.282050,3.261966,0.132279,0.141025,0.130983


### Assert mutual absolute scale --> Check by revert back to its previous form 

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To check this, I implement the revert method to vac-bert scale and assert it to contain the same value for original scale then there is unchanged in absolute meaning in value

- The output must be 0

In [11]:
# map 1..5 -> -1..1
def one5_to_minus1_1(x, xmin=1.0, xmax=5.0):
    x01 = (x - xmin) / (xmax - xmin)
    return 2*x01 - 1

def minus1_1_to_one5(y, xmin=1.0, xmax=5.0):
    x01 = (y + 1) / 2
    return x01 * (xmax - xmin) + xmin

diff = df["arousal_text"] - minus1_1_to_one5(df["arousal_text_n"])
print(diff.abs().max())   # ควร ~ 0 (มีแค่ numerical noise ระดับ 1e-7)


0.0


### Check rank & order value

- The output must be 1

In [12]:
# index ของค่า arousal สูงสุด ก่อนและหลัง normalize ต้องเป็นอันเดียวกัน
orig_argmax = df["arousal_text"].idxmax()
norm_argmax = df["arousal_text_n"].idxmax()
print(orig_argmax, norm_argmax)

# หรือ correlation ระหว่างค่าเดิมกับค่าที่ normalize ควร = 1
df[["arousal_text", "arousal_text_n"]].corr()


0 0


,arousal_text,arousal_text_n
arousal_text,1.0,1.0
arousal_text_n,1.0,1.0


### Check distribution

In [13]:
print(df["arousal_text"].describe())
print(df["arousal_text_n"].describe())


count    6.000000
mean     3.541567
std      0.245863
min      3.282050
25%      3.380994
50%      3.507132
75%      3.608547
max      3.971787
Name: arousal_text, dtype: float64
count    6.000000
mean     0.270783
std      0.122932
min      0.141025
25%      0.190497
50%      0.253566
75%      0.304274
max      0.485893
Name: arousal_text_n, dtype: float64


### Save to .csv format

In [14]:
df.to_csv("../own_script/dialogue_1/dialogue_1_vad_text.csv", index=False)

## Compare with wagner --> is it related to each other?

### Implement wagner speech module

In [15]:
import pandas as pd

# 1) โหลด speech จาก wagner
speech_df = pd.read_csv("../wagner/emotion_results_w2v2_own.csv")
print("speech rows:", len(speech_df))
print(speech_df.head())

# 2) โหลด text จาก dialogue_1.txt
with open("../own_script/dialogue_1/dialogue_1_vad_text.csv", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]

text_df = pd.DataFrame({
    "utterance_id": range(1, len(lines)+1),
    "text": lines,
})
print("text rows:", len(text_df))
print(text_df.head())


speech rows: 6
                     filename  dialogue_id  utterance_id   arousal  dominance  \
0  dialogue_1_utterance_1.wav            1             1  0.010609   0.014264   
1  dialogue_1_utterance_2.wav            1             2  0.002821   0.000275   
2  dialogue_1_utterance_3.wav            1             3 -0.000294  -0.008163   
3  dialogue_1_utterance_4.wav            1             4 -0.001352  -0.004913   
4  dialogue_1_utterance_5.wav            1             5 -0.001965   0.011020   

    valence  
0 -0.017251  
1 -0.012718  
2 -0.019781  
3 -0.002546  
4 -0.006344  
text rows: 7
   utterance_id                                               text
0             1  utterance_id,text,valence_text,arousal_text,do...
1             2  1,I’m so tired! This player just keeps running...
2             3  2,Honestly? A solid 9. It’s boring and feels l...
3             4  3,"Both! A good player should stand their grou...
4             5  4,You mean you're saying they're running becau...

In [ ]:
import pandas as pd

speech_df = pd.read_csv("../wagner/emotion_results_w2v2_own.csv")  # ที่เราสร้างไว้
speech_d10 = speech_df[speech_df["dialogue_id"] == 1].sort_values("utterance_id")

text_df = pd.read_csv("dialogue_10_vad_text.csv").sort_values("utterance_id")


### Check if each utterance aligned well with plot

### Find right value of delta (T)

In [17]:
import pandas as pd
import numpy as np

# 1) เตรียมให้ utterance_id ตรงกัน
speech = speech_df[speech_df["dialogue_id"] == 1].sort_values("utterance_id").reset_index(drop=True)
text   = text_df.sort_values("utterance_id").reset_index(drop=True)
assert (speech["utterance_id"].to_numpy() == text["utterance_id"].to_numpy()).all()

# 2) speech: ใช้ค่าจาก Wagner ตรง ๆ (สมมติอยู่ใน [-1,1] แล้ว)
speech_norm = pd.DataFrame({
    "utterance_id": speech["utterance_id"],
    "val_s": speech["valence"].to_numpy(),
    "aro_s": speech["arousal"].to_numpy(),
    "dom_s": speech["dominance"].to_numpy(),
})

# 3) text: normalize 1..5 -> -1..1
V_MIN, V_MAX = 1.0, 5.0

def one5_to_minus1_1(x, xmin=V_MIN, xmax=V_MAX):
    x01 = (x - xmin) / (xmax - xmin)   # 1..5 -> 0..1
    return 2.0 * x01 - 1.0            # 0..1 -> -1..1

text_norm = pd.DataFrame({
    "utterance_id": text["utterance_id"],
    "val_t": one5_to_minus1_1(text["valence_text"].to_numpy()),
    "aro_t": one5_to_minus1_1(text["arousal_text"].to_numpy()),
    "dom_t": one5_to_minus1_1(text["dominance_text"].to_numpy()),
})

# 4) รวม แล้วคำนวณ delta
df = speech_norm.merge(text_norm, on="utterance_id")

df["delta_valence"]   = (df["val_s"] - df["val_t"]).abs()
df["delta_arousal"]   = (df["aro_s"] - df["aro_t"]).abs()
df["delta_dominance"] = (df["dom_s"] - df["dom_t"]).abs()

# 5) threshold บน absolute scale [-1,1]
VAL_THR = 0.5
ARO_THR = 0.5
DOM_THR = 0.5

df["dissonant_valence"]   = df["delta_valence"]   > VAL_THR
df["dissonant_arousal"]   = df["delta_arousal"]   > ARO_THR
df["dissonant_dominance"] = df["delta_dominance"] > DOM_THR

df["dissonant_any"] = (
    df["dissonant_valence"] |
    df["dissonant_arousal"] |
    df["dissonant_dominance"]
)

df.head(10)


,utterance_id,val_s,aro_s,dom_s,val_t,aro_t,dom_t,delta_valence,delta_arousal,delta_dominance,dissonant_valence,dissonant_arousal,dissonant_dominance,dissonant_any
0,1,-0.017251,0.010609,0.014264,-0.288250,0.485893,-0.097817,0.270999,0.475285,0.112082,False,False,False,False
1,2,-0.012718,0.002821,0.000275,-0.229381,0.316803,0.013185,0.216663,0.313982,0.012909,False,False,False,False
2,3,-0.019781,-0.000294,-0.008163,-0.080103,0.266685,0.304801,0.060321,0.266979,0.312964,False,False,False,False
3,4,-0.002546,-0.001352,-0.004913,-0.128125,0.240447,0.032955,0.125579,0.241798,0.037868,False,False,False,False
4,5,-0.006344,-0.001965,0.011020,-0.148869,0.173847,0.147515,0.142525,0.175813,0.136495,False,False,False,False
5,6,-0.013262,-0.007406,0.001734,0.132279,0.141025,0.130983,0.145541,0.148431,0.129249,False,False,False,False
